# Superfícies e sólidos esféricos

**Módulo 06 — Geometria Espacial** · Notebook 18 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0618a_superficies_e_solidos_esfericos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (seções, anticlépsidra, fuso e cunha), [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (tronco de cone) e [`0617a_superficies_e_solidos_de_revolucao.ipynb`](0617a_superficies_e_solidos_de_revolucao.ipynb) (plano meridiano, Pappus-Guldin e discos empilhados)
- Módulo 04 — Geometria Plana: [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (relações métricas no triângulo retângulo)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar zonas e calotas esféricas e calcular suas áreas;
- identificar segmentos, setores e anéis esféricos;
- deduzir e usar as fórmulas dos volumes desses sólidos;
- resolver problemas reais com partes da esfera, como a parte da Terra visível de uma certa altura.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Quanto da Terra você consegue ver?

Do alto de um prédio, você vê o horizonte a alguns quilômetros. De um avião, bem mais longe. Os astronautas da Estação Espacial Internacional, a cerca de 400 km de altitude, veem continentes inteiros. Mas qual **fração** da superfície da Terra cada um deles enxerga?

A região visível é uma "tampa" da esfera, chamada **calota esférica**. Ela é limitada pelos pontos onde a linha de visão **tangencia** a Terra.

🕹️ No widget abaixo, escolha onde está o observador e veja a parte visível da Terra. (A fórmula usada será explicada na seção 2.)

In [2]:
# Cores usadas em todo o notebook: cada papel tem a mesma cor na figura 3D e na
# figura plana (o gráfico 2D logo abaixo de cada figura 3D).
COR_ESFERA = u3d.CORES["apoio"]              # a esfera inteira, quase transparente
COR_PARTE = u3d.CORES["destaque"]            # a zona, a calota ou a região que gira
COR_SOLIDO = u3d.CORES["solido"]             # o sólido esférico
COR_PLANOS = u3d.CORES["plano"]              # os planos de corte
COR_EXTRA = u3d.CORES["plano_secundario"]    # observador, troncos, centroide etc.


def latitude(z: float, raio: float) -> float:
    """Latitude (em graus) dos pontos da esfera na altura z: o ângulo cujo seno é z/R."""
    seno = max(-1, min(1, z / raio))   # protege contra arredondamentos fora de [-1, 1]
    return math.degrees(math.asin(seno))


def criar_plano_meridiano(titulo: str):
    """Cria o gráfico 2D do plano meridiano: r na horizontal e z na vertical."""
    return u3d.criar_grafico(
        titulo=titulo, rotulo_x="r (distância ao eixo)", rotulo_y="z (altura)",
        proporcao_igual=True,
    )


def desenhar_esfera_no_plano(grafico, raio: float, alturas_dos_planos=()):
    """Desenha no plano meridiano o eixo, a semicircunferência de raio R (tracejada)
    e os planos de corte, que aparecem como linhas horizontais."""
    t = np.linspace(-math.pi / 2, math.pi / 2, 100)
    grafico = u3d.desenhar_curva(
        grafico, raio * np.cos(t), raio * np.sin(t), rotulo=f"esfera de raio {raio:g}",
        cor=COR_ESFERA, espessura=2, tracejado=True,
    )
    grafico = u3d.desenhar_curva(
        grafico, [0, 0], [-1.15 * raio, 1.15 * raio], rotulo="eixo", cor=COR_ESFERA,
        tracejado=True,
    )
    for z in alturas_dos_planos:
        grafico = u3d.desenhar_curva(
            grafico, [0, 1.2 * raio], [z, z], rotulo=f"plano z = {z:g}", cor=COR_PLANOS,
            espessura=2,
        )
    return grafico


def arco_no_plano(raio: float, z_inferior: float, z_superior: float, n: int = 100):
    """Pontos (r, z) do arco da circunferência de raio R entre duas alturas."""
    t = np.radians(np.linspace(latitude(z_inferior, raio), latitude(z_superior, raio), n))
    return raio * np.cos(t), raio * np.sin(t)


# Raio da Terra, em km.
R_TERRA = 6371

# Cada observador: (altitude real em km, altitude usada só no desenho, com a Terra
# de raio 1). O desenho fica FORA DE ESCALA: em escala, o prédio e o avião ficariam
# colados na Terra, e a Lua ficaria a 60 raios de distância, longe demais para caber
# na figura. Os números impressos usam as medidas reais.
OBSERVADORES = {
    "Prédio alto (0,5 km)": (0.5, 0.04),
    "Avião (10 km)": (10, 0.12),
    "Estação Espacial (400 km)": (400, 0.35),
    "Satélite geoestacionário (35 786 km)": (35786, 1.5),
    "Lua (384 400 km)": (384400, 3),
}


def ver_a_terra(observador: str) -> None:
    """Desenha a calota visível pelo observador e calcula a porcentagem visível."""
    altitude, altitude_no_desenho = OBSERVADORES[observador]

    # Os cálculos, com as medidas reais (a fórmula é explicada na seção 2).
    R = R_TERRA
    H = altitude
    altura_da_calota = R * H / (R + H)        # explicada na seção 2
    fracao_visivel = altura_da_calota / (2 * R)
    porcentagem = fracao_visivel * 100

    # O desenho, com a Terra de raio 1. O plano da borda da calota fica a uma
    # distância d do centro (seção 2) e corta a Terra numa circunferência.
    distancia = 1 + altitude_no_desenho       # do centro da Terra até o observador
    d = 1 / distancia                          # do centro até o plano da borda
    raio_da_borda = math.sqrt(1 - d**2)
    olho = (0, 0, distancia)

    fig = u3d.criar_figura(titulo=f"{observador} (desenho fora de escala)")
    fig = u3d.desenhar_esfera(fig, (0, 0, 0), 1, cor=COR_ESFERA, opacidade=0.2)
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), 1, cor=COR_PARTE, opacidade=0.7,
        latitude_minima=latitude(d, 1), latitude_maxima=90,
    )
    fig = u3d.desenhar_circunferencia(fig, (0, 0, d), raio_da_borda, cor=COR_PARTE)
    # Algumas linhas de visão: do observador até a borda da calota, tangentes à Terra.
    for k in range(8):
        t = 2 * math.pi * k / 8
        borda = (raio_da_borda * math.cos(t), raio_da_borda * math.sin(t), d)
        fig = u3d.desenhar_segmento(fig, olho, borda, cor=COR_EXTRA, espessura=2)
    fig = u3d.desenhar_ponto(fig, olho, rotulo="observador", cor=COR_EXTRA)
    u3d.mostrar(fig)

    # Figura plana: o corte pelo centro da Terra e pelo observador.
    grafico = u3d.criar_grafico(
        titulo="Corte pelo centro da Terra e pelo observador", proporcao_igual=True
    )
    grafico = u3d.desenhar_circunferencia_2d(
        grafico, (0, 0), 1, rotulo="Terra", cor=COR_ESFERA
    )
    t = np.linspace(math.asin(d), math.pi - math.asin(d), 100)   # o arco visível
    grafico = u3d.desenhar_curva(
        grafico, np.cos(t), np.sin(t), rotulo="parte visível", cor=COR_PARTE, espessura=5
    )
    for lado in (-1, 1):
        grafico = u3d.desenhar_curva(
            grafico, [0, lado * raio_da_borda], [distancia, d],
            rotulo="linha de visão (tangente)" if lado == 1 else None, cor=COR_EXTRA,
        )
    grafico = u3d.desenhar_curva(
        grafico, [-raio_da_borda, raio_da_borda], [d, d], rotulo="base da calota",
        cor=COR_PLANOS, tracejado=True,
    )
    grafico = u3d.marcar_ponto_no_grafico(
        grafico, 0, distancia, rotulo="observador", cor=COR_EXTRA
    )
    u3d.mostrar(grafico)

    print(f"Altura da calota visível: {altura_da_calota:.2f} km")
    if porcentagem >= 0.1:
        print(f"Porcentagem visível da superfície da Terra: {porcentagem:.2f}%")
    else:   # porcentagens muito pequenas precisam de mais casas decimais
        print(f"Porcentagem visível da superfície da Terra: {porcentagem:.4f}%")
    if observador.startswith("Lua"):
        print("Mesmo da Lua, muito longe, não se vê a metade da Terra: a porcentagem")
        print("chega perto de 50%, mas nunca chega lá.")


interact(
    ver_a_terra,
    observador=widgets.Dropdown(
        options=list(OBSERVADORES), value="Estação Espacial (400 km)",
        description="observador", style={"description_width": "initial"},
        layout=widgets.Layout(width="420px"),
    ),
);

interactive(children=(Dropdown(description='observador', index=2, layout=Layout(width='420px'), options=('Préd…

Do avião, você vê menos de **0,1%** da Terra. Os astronautas veem cerca de **3%**. Um satélite geoestacionário vê cerca de **42%**, e é por isso que três deles bastam para cobrir quase todo o planeta com sinais de comunicação. E nem da Lua se vê a metade da Terra: chega-se perto de 50%, mas nunca lá.

Neste último notebook do módulo, você vai estudar as partes da esfera, suas áreas e seus volumes, usando praticamente todas as ferramentas que aprendeu até aqui.

Como no [notebook 17](0617a_superficies_e_solidos_de_revolucao.ipynb), as figuras planas mostram o **plano meridiano**: $r$ é a distância até o eixo e $z$ é a altura. A esfera tem o centro na origem, eixo vertical e raio $R$; as partes ficam entre os planos horizontais $z = z_1$ e $z = z_2$, e $h = z_2 - z_1$ é a **altura**. Os raios das bases são $a = \sqrt{R^2 - z_1^2}$ e $b = \sqrt{R^2 - z_2^2}$ (as seções da esfera, [notebook 13](0613a_esfera.ipynb)).

## 1. Superfícies esféricas: zona e calota

> 📌 **Definição.**
> - Uma **zona esférica** é a parte da superfície esférica compreendida entre **dois planos paralelos** que cortam a esfera. Ela tem **duas bases** (as circunferências de corte), e a distância $h$ entre os planos é a sua **altura**.
> - Uma **calota esférica** é a parte da superfície esférica cortada por **um** plano: de um lado do plano. Ela tem **uma base**, e a altura $h$ é a distância do plano até o ponto mais afastado da calota (o "polo" dela).

Uma calota é uma zona em que um dos planos é **tangente** à esfera. E lembre: o **fuso** ([notebook 13](0613a_esfera.ipynb)) é outra parte da superfície, limitada por dois **semiplanos** que passam pelo eixo, e não por planos paralelos.

**No dia a dia:** a parte da Terra entre dois paralelos (por exemplo, a zona tropical, entre os trópicos, é uma zona esférica), a calota polar, a tampa de uma bola cortada, a cúpula de um observatório.

🕹️ Escolha os planos. Com `z_superior` = 2 (ou `z_inferior` = −2), o plano fica tangente à esfera e a zona vira uma calota.

In [3]:
R = 2   # o raio da esfera das seções 1 a 3


def mostrar_zona(z_inferior: float, z_superior: float) -> None:
    """Desenha a zona (ou calota) entre os dois planos e mede a altura e as bases."""
    if z_inferior >= z_superior:
        print("Escolha z_inferior menor que z_superior: o plano de baixo deve ficar")
        print("abaixo do plano de cima.")
        return

    h = z_superior - z_inferior
    raio_base_inferior = math.sqrt(R**2 - z_inferior**2)
    raio_base_superior = math.sqrt(R**2 - z_superior**2)
    tipo = "calota" if (z_superior == R or z_inferior == -R) else "zona"
    if z_superior == R and z_inferior == -R:
        tipo = "superfície esférica inteira (os dois planos são tangentes)"

    fig = u3d.criar_figura(titulo=f"A {tipo} entre os planos")
    fig = u3d.desenhar_esfera(fig, (0, 0, 0), R, cor=COR_ESFERA, opacidade=0.12)
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), R, cor=COR_PARTE, opacidade=0.7,
        latitude_minima=latitude(z_inferior, R), latitude_maxima=latitude(z_superior, R),
    )
    for z, nome, raio_da_base in [
        (z_inferior, "α", raio_base_inferior), (z_superior, "β", raio_base_superior)
    ]:
        fig = u3d.desenhar_plano(
            fig, (0, 0, z), (0, 0, 1), rotulo=nome, cor=COR_PLANOS, opacidade=0.1,
            tamanho=5.5,
        )
        if raio_da_base > 0:   # num plano tangente, a "base" é só um ponto
            fig = u3d.desenhar_circunferencia(fig, (0, 0, z), raio_da_base, cor=COR_PARTE)
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("O arco que gera a zona, no plano meridiano")
    grafico = desenhar_esfera_no_plano(grafico, R, [z_inferior, z_superior])
    rs, zs = arco_no_plano(R, z_inferior, z_superior)
    grafico = u3d.desenhar_curva(grafico, rs, zs, rotulo=tipo.split()[0], cor=COR_PARTE, espessura=5)
    u3d.mostrar(grafico)

    print(f"Tipo: {tipo}")
    print(f"Altura: h = {h:g}")
    print(f"Raio da base de baixo: {raio_base_inferior:.4f}")
    print(f"Raio da base de cima:  {raio_base_superior:.4f}")


interact(
    mostrar_zona,
    z_inferior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=0.5, description="z_inferior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    z_superior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=2, description="z_superior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='z_inferior', max=2.0, min=-…

**Pare e pense**

1. A superfície de uma semiesfera é uma zona ou uma calota? Qual é a altura dela?
2. Os paralelos de latitude 30° N e 30° S limitam uma zona esférica na Terra. Qual é a altura dessa zona, em função de $R$?

<details>
<summary>Ver respostas</summary>

1. Uma calota (o plano do equador corta, e do outro lado só há o polo), com altura $R$.
2. Cada paralelo fica a uma distância $R\sin 30° = \frac{R}{2}$ do plano do equador, um de cada lado. A altura é $R$.
</details>

## 2. Área da zona e da calota

**Teorema**
> A área de uma zona esférica (ou de uma calota) de altura $h$, numa esfera de raio $R$, é
> $$\boxed{S = 2\pi R h}$$

Repare numa coisa surpreendente: a área **só depende da altura** $h$ (e do raio da esfera), e **não** de onde a zona está! Uma faixa de altura 1 perto do equador e outra de altura 1 perto do polo têm a **mesma área**: a de perto do polo é mais estreita, mas é mais "inclinada".

> 🕵️ **Curiosidade histórica.** Esse resultado também é de **Arquimedes**. Ele mostrou que, se a esfera estiver dentro do cilindro que a envolve (raio $R$, altura $2R$), cada zona esférica tem a mesma área que a faixa do cilindro entre os mesmos dois planos: $2\pi R \cdot h$. Com $h = 2R$, voltamos à área da esfera inteira, $4\pi R^2$.

> 💡 **Dica.** Essa ideia de Arquimedes deu origem a um tipo de **mapa** (a projeção cilíndrica equivalente, de Lambert): projetando cada ponto da Terra horizontalmente sobre o cilindro que a envolve e depois desenrolando o cilindro, obtém-se um mapa em que **todas as regiões mantêm as suas áreas** (embora as formas fiquem distorcidas perto dos polos).

### Por que $2\pi R h$? Uma aproximação por troncos de cone

Divida o arco que gera a zona em vários pedacinhos retos. Cada pedacinho, girando, gera a superfície lateral de um **tronco de cone**, cuja área é $\pi(r_1 + r_2)\,g$ ([notebook 14](0614a_solidos_semelhantes_e_troncos.ipynb)). Somando as áreas desses troncos, obtemos uma aproximação da área da zona, que fica melhor quanto mais pedacinhos usarmos. O widget abaixo mostra que essa soma se aproxima de $2\pi R h$.

Há também uma explicação geométrica bonita: num pedacinho do arco, o triângulo formado pelo pedacinho e pela sua "sombra" no eixo é **semelhante** ao triângulo formado pelo raio, então (distância ao eixo) × (comprimento do pedacinho) = $R$ × (sombra no eixo). Somando, todos os pedacinhos juntos dão $R \cdot h$, e a área fica $2\pi R h$.

### Calota: outra fórmula

Numa calota de altura $h$ e raio da base $a$, a seção meridiana dá $a^2 = h(2R - h)$, ou seja, $2Rh = a^2 + h^2$. Então:
$$S_{\text{calota}} = \pi(a^2 + h^2) = \pi \ell^2$$
em que $\ell = \sqrt{a^2 + h^2}$ é a **distância do polo da calota até a borda da base**. A área da calota é a de um **círculo** com esse raio!

### A resposta do gancho

O observador está a uma altura $H$ acima da superfície, ou seja, a uma distância $R + H$ do centro $O$. A linha de visão tangencia a Terra num ponto $T$, e $OT \perp PT$ (o raio é perpendicular à tangente, [notebook 13](0613a_esfera.ipynb)). No triângulo retângulo $OTP$, a projeção de $OT$ sobre a hipotenusa $OP$ dá a distância $d$ do centro ao plano da base da calota (relações métricas, [módulo 04](../04_geometria_plana/0416a_triangulos_retangulos.ipynb)):
$$R^2 = d \cdot (R + H) \quad\Longrightarrow\quad d = \frac{R^2}{R + H} \quad\Longrightarrow\quad h = R - d = \frac{RH}{R + H}$$

E a fração visível é $\dfrac{2\pi R h}{4\pi R^2} = \dfrac{h}{2R}$.

🕹️ Escolha o modo. No modo "Aproximação por troncos de cone", valem os controles `z_inferior`, `z_superior` e `pedacos`; no modo "Mesma altura, lugares diferentes", valem `altura_da_faixa` e `posicao` (a altura onde começa a segunda faixa).

In [4]:
def somar_troncos(z_inferior: float, z_superior: float, pedacos: int):
    """Soma as áreas laterais dos troncos de cone gerados pelos pedaços retos do arco
    (esfera de raio R). Devolve a soma e os pontos do arco usados."""
    zs = np.linspace(z_inferior, z_superior, pedacos + 1)
    pontos = [(math.sqrt(max(0, R**2 - z**2)), z) for z in zs]   # pontos do arco
    soma = 0
    for i in range(pedacos):
        (r1, z1), (r2, z2) = pontos[i], pontos[i + 1]
        g = math.sqrt((r2 - r1)**2 + (z2 - z1)**2)
        soma = soma + math.pi * (r1 + r2) * g                     # área lateral do tronco de cone
    return soma, pontos


def aproximar_por_troncos(z_inferior: float, z_superior: float, pedacos: int) -> None:
    """Modo 1: a zona e os troncos de cone que a aproximam."""
    if z_inferior >= z_superior:
        print("Escolha z_inferior menor que z_superior.")
        return
    soma, pontos = somar_troncos(z_inferior, z_superior, pedacos)
    area_exata = 2 * math.pi * R * (z_superior - z_inferior)

    fig = u3d.criar_figura(titulo=f"A zona e {pedacos} troncos de cone")
    fig = u3d.desenhar_esfera(
        fig, (0, 0, 0), R, cor=COR_PARTE, opacidade=0.15,
        latitude_minima=latitude(z_inferior, R), latitude_maxima=latitude(z_superior, R),
    )
    for i in range(pedacos):
        # Cada pedaço reto do arco, girando, gera a superfície lateral de um tronco de
        # cone (ou de um cone, se tocar o eixo). As cores alternam, para separá-los.
        cor = COR_SOLIDO if i % 2 == 0 else COR_EXTRA
        fig = u3d.desenhar_superficie_de_revolucao(
            fig, [pontos[i], pontos[i + 1]], cor=cor, opacidade=0.7, mostrar_perfil=False
        )
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("O arco e os pedaços retos")
    grafico = desenhar_esfera_no_plano(grafico, R, [z_inferior, z_superior])
    rs, zs = arco_no_plano(R, z_inferior, z_superior)
    grafico = u3d.desenhar_curva(grafico, rs, zs, rotulo="arco", cor=COR_PARTE, espessura=5)
    grafico = u3d.desenhar_curva(
        grafico, [r for r, z in pontos], [z for r, z in pontos], rotulo="pedaços retos",
        cor=COR_EXTRA,
    )
    u3d.mostrar(grafico)

    print(f"Soma das áreas dos {pedacos} troncos: {soma:.4f}")
    print(f"Área exata, 2πRh:              {area_exata:.4f}")
    print(f"Diferença:                     {soma - area_exata:+.4f}")
    print()
    print("pedaços |       soma")
    for n in [1, 2, 5, 10, 50, 100]:
        soma_n, _ = somar_troncos(z_inferior, z_superior, n)
        print(f"{n:7d} | {soma_n:10.4f}")


def comparar_faixas(altura_da_faixa: float, posicao: float) -> None:
    """Modo 2: duas faixas de mesma altura, uma no equador e outra mais acima."""
    faixas = [(-altura_da_faixa / 2, altura_da_faixa / 2)]       # centrada no equador
    if posicao + altura_da_faixa > R:
        posicao = R - altura_da_faixa   # a faixa não pode passar do polo norte
        print(f"(A segunda faixa foi descida para começar em z = {posicao:g}, para caber na esfera.)")
    faixas.append((posicao, posicao + altura_da_faixa))
    cores = [COR_PARTE, COR_EXTRA]
    nomes = ["faixa no equador", "faixa na posição escolhida"]

    fig = u3d.criar_figura(titulo=f"Duas faixas de altura {altura_da_faixa:g}")
    fig = u3d.desenhar_esfera(fig, (0, 0, 0), R, cor=COR_ESFERA, opacidade=0.12)
    grafico = criar_plano_meridiano("As duas faixas no plano meridiano")
    grafico = desenhar_esfera_no_plano(grafico, R)
    for (z1, z2), cor, nome in zip(faixas, cores, nomes):
        fig = u3d.desenhar_esfera(
            fig, (0, 0, 0), R, cor=cor, opacidade=0.75,
            latitude_minima=latitude(z1, R), latitude_maxima=latitude(z2, R),
        )
        rs, zs = arco_no_plano(R, z1, z2)
        grafico = u3d.desenhar_curva(grafico, rs, zs, rotulo=nome, cor=cor, espessura=6)
    u3d.mostrar(fig)
    u3d.mostrar(grafico)

    for (z1, z2), nome in zip(faixas, nomes):
        area, _ = somar_troncos(z1, z2, 200)
        z_meio = (z1 + z2) / 2
        raio_no_meio = math.sqrt(R**2 - z_meio**2)
        # O comprimento do arco que gera a faixa: raio × ângulo (em radianos).
        arco = R * math.radians(latitude(z2, R) - latitude(z1, R))
        print(f"{nome}: z de {z1:g} a {z2:g}")
        print(f"   área (200 troncos) = {area:.4f}   raio no meio = {raio_no_meio:.4f}"
              f"   comprimento do arco = {arco:.4f}")
    print(f"Área exata das duas, 2πRh = {2 * math.pi * R * altura_da_faixa:.4f}")
    print("Perto do polo, a faixa é mais estreita (raio menor), mas o arco é mais comprido")
    print("(a faixa é mais inclinada): as duas coisas se compensam exatamente.")


MODOS = ["Aproximação por troncos de cone", "Mesma altura, lugares diferentes"]


def explorar_areas(modo, z_inferior, z_superior, pedacos, altura_da_faixa, posicao) -> None:
    """Escolhe o modo do widget."""
    if modo == MODOS[0]:
        aproximar_por_troncos(z_inferior, z_superior, pedacos)
    else:
        comparar_faixas(altura_da_faixa, posicao)


interact(
    explorar_areas,
    modo=widgets.RadioButtons(options=MODOS, description="modo"),
    z_inferior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=0.5, description="z_inferior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    z_superior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=2, description="z_superior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    pedacos=widgets.IntSlider(
        min=1, max=30, step=1, value=4, description="pedacos",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    altura_da_faixa=widgets.FloatSlider(
        min=0.25, max=1, step=0.25, value=0.5, description="altura_da_faixa",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    posicao=widgets.FloatSlider(
        min=-2, max=1.75, step=0.25, value=1.25, description="posicao",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(RadioButtons(description='modo', options=('Aproximação por troncos de cone', 'Mesma altu…

**Pare e pense**

1. Numa esfera de raio 5, qual é a área de uma zona de altura 2? E de uma calota de altura 2?
2. Uma calota tem base de raio 4 e altura 2. Qual é a sua área? E o raio da esfera?

<details>
<summary>Ver respostas</summary>

1. As duas têm área $2\pi \cdot 5 \cdot 2 = 20\pi \approx 62{,}8$.
2. $S = \pi(16 + 4) = 20\pi$. De $2Rh = a^2 + h^2$: $4R = 20$, então $R = 5$.
</details>

## 3. Sólidos esféricos

Assim como a superfície esférica se divide em zonas, calotas e fusos, a **esfera** (o sólido) se divide em partes. Todas elas são **sólidos de revolução** ([notebook 17](0617a_superficies_e_solidos_de_revolucao.ipynb)), gerados por uma região do plano meridiano girando em torno do eixo.

> 📌 **Definição.**
> - O **segmento esférico** é a parte da esfera compreendida entre dois planos paralelos (**de duas bases**) ou cortada por um único plano (**de uma base**, como uma "tampa" maciça). Sua superfície curva é uma zona ou uma calota.
> - O **setor esférico** é o sólido gerado por um **setor circular** (com vértice no centro da esfera) girando em torno de um diâmetro que não o atravessa. Ele é formado por uma zona ou calota "ligada" ao centro por superfícies cônicas, como um "cone de sorvete" com tampa esférica.
> - O **anel esférico** é o sólido gerado por um **segmento circular** (a região entre um arco e a sua corda) girando em torno de um diâmetro que não o atravessa.
> - A **cunha esférica**, você já conhece do [notebook 13](0613a_esfera.ipynb): é a parte da esfera entre dois semiplanos que passam pelo eixo.

**No dia a dia:** uma fatia de melancia cortada por dois planos paralelos (segmento de duas bases), uma "tampa" de laranja (segmento de uma base), a forma de uma casquinha com sorvete arredondado em cima (setor esférico), um anel de guardanapo arredondado (anel esférico).

🕹️ Escolha o sólido e gire a região. No segmento de uma base, o plano de cima é tangente à esfera no polo norte, e o controle `z_superior` é ignorado.

In [5]:
# A função do utils_3d que devolve a região do plano meridiano de cada sólido.
REGIOES = {
    "Segmento de uma base": u3d.regiao_meridiana_do_segmento,
    "Segmento de duas bases": u3d.regiao_meridiana_do_segmento,
    "Setor esférico": u3d.regiao_meridiana_do_setor,
    "Anel esférico": u3d.regiao_meridiana_do_anel,
}


def girar_parte_da_esfera(solido: str, z_inferior: float, z_superior: float, giro: int) -> None:
    """Gira a região do plano meridiano e mostra o sólido esférico gerado."""
    if solido == "Segmento de uma base":
        z_superior = R   # o plano de cima é tangente no polo norte
    if z_inferior >= z_superior:
        print("Escolha z_inferior menor que z_superior.")
        return
    regiao = REGIOES[solido](R, z_inferior, z_superior, resolucao=100)

    fig = u3d.criar_figura(titulo=f"{solido}: giro de {giro}°")
    fig = u3d.desenhar_esfera(fig, (0, 0, 0), R, cor=COR_ESFERA, opacidade=0.08)
    fig = u3d.desenhar_solido_de_revolucao(
        fig, regiao, angulo=giro, cor=COR_SOLIDO, opacidade=0.6
    )
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("A região que gira, no plano meridiano")
    grafico = desenhar_esfera_no_plano(grafico, R, [z_inferior, z_superior])
    grafico = u3d.desenhar_poligono_2d(grafico, regiao, rotulo="região", cor=COR_PARTE)
    u3d.mostrar(grafico)


interact(
    girar_parte_da_esfera,
    solido=widgets.Dropdown(
        options=list(REGIOES), description="sólido",
        style={"description_width": "initial"},
    ),
    z_inferior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=0.5, description="z_inferior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    z_superior=widgets.FloatSlider(
        min=-2, max=2, step=0.25, value=1.5, description="z_superior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    giro=widgets.IntSlider(
        min=30, max=360, step=30, value=270, description="giro (graus)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='sólido', options=('Segmento de uma base', 'Segmento de duas bases'…

**Pare e pense**

1. Que região, girando, gera um setor esférico cuja superfície curva é uma calota?
2. Qual é a diferença entre um segmento esférico de duas bases e um anel esférico gerados entre os mesmos dois planos?

<details>
<summary>Ver respostas</summary>

1. Um setor circular com um dos raios sobre o eixo. O sólido é formado por um cone (com vértice no centro) e um segmento esférico de uma base.
2. O segmento ocupa toda a "fatia" da esfera entre os planos, até o eixo. O anel é só a parte de fora da corda: é o segmento menos o tronco de cone (ou cilindro) gerado pela região entre a corda e o eixo.
</details>

## 4. Volumes dos sólidos esféricos

### Segmento esférico de uma base (Cavalieri e anticlépsidra)

No [notebook 13](0613a_esfera.ipynb), você viu que a esfera e a anticlépsidra (o cilindro equilátero menos dois cones) têm seções de mesma área em **todas** as alturas. Então, entre dois planos quaisquer, as **fatias** dos dois sólidos também têm o mesmo volume (Princípio de Cavalieri, [notebook 09](0609a_volume_e_principio_de_cavalieri.ipynb)).

Tome o segmento de uma base no topo da esfera, de altura $h$ (entre $z = R - h$ e $z = R$). A parte correspondente da anticlépsidra é uma fatia de cilindro de raio $R$ e altura $h$, **menos** um **tronco de cone** de raios $R$ e $R - h$ e altura $h$ ([notebook 14](0614a_solidos_semelhantes_e_troncos.ipynb)):
$$\text{volume} = \pi R^2 h - \frac{\pi h}{3}\left(R^2 + R(R - h) + (R - h)^2\right)$$

Simplificando:
$$\boxed{\text{volume do segmento de uma base} = \frac{\pi h^2}{3}(3R - h)}$$

Também é útil escrever em função do raio da base $a$ (usando $a^2 = h(2R - h)$):
$$\text{volume} = \frac{\pi h}{6}\left(3a^2 + h^2\right)$$

### Segmento esférico de duas bases

É a diferença entre dois segmentos de uma base. Com raios das bases $a$ e $b$ e altura $h$, chega-se a:
$$\boxed{\text{volume do segmento de duas bases} = \frac{\pi h}{6}\left(3a^2 + 3b^2 + h^2\right)}$$

### Setor esférico ("pirâmides" com vértice no centro)

Como no [notebook 13](0613a_esfera.ipynb), pense no setor dividido em muitas "pirâmides" finas com vértice no centro e base na zona (ou calota), todas com altura $R$. Então o volume é $\frac{1}{3} \cdot (\text{área da zona}) \cdot R$:
$$\boxed{\text{volume do setor} = \frac{1}{3} \cdot 2\pi R h \cdot R = \frac{2}{3}\pi R^2 h}$$

Com $h = 2R$ (a zona é a esfera inteira), voltamos a $\frac{4}{3}\pi R^3$. ✔

### Anel esférico (Pappus)

O anel é o setor esférico **menos** o sólido gerado pelo **triângulo** formado pelo centro e pela corda. Esse triângulo tem altura $m$ (a distância do centro à corda), e o seu volume de revolução sai por Pappus ([notebook 17](0617a_superficies_e_solidos_de_revolucao.ipynb)). Fazendo as contas, aparece uma fórmula surpreendentemente simples, em que o raio da esfera **desaparece**:
$$\boxed{\text{volume do anel} = \frac{\pi \ell^2 h}{6}}$$
em que $\ell$ é o comprimento da corda e $h$ é a sua projeção no eixo (a altura do anel).

> 💡 **Dica.** Fure uma esfera com um furo cilíndrico, passando pelo centro, de modo que o anel que sobra tenha altura $h$. A corda que gera esse anel é vertical, então $\ell = h$ e o volume que sobra é $\frac{\pi h^3}{6}$, **qualquer que seja o tamanho da esfera**! Uma esfera do tamanho de uma laranja e outra do tamanho da Terra, furadas de modo a sobrar um anel de 6 cm de altura, deixam o mesmo volume. Esse é o famoso "problema do anel de guardanapo".

🕹️ Confira as fórmulas com Pappus e com discos. O controle `discos` vale só para os segmentos. Se uma altura passar do raio, ela é trazida de volta para dentro da esfera.

In [6]:
def conferir_volume(
    solido: str, raio: float, z_inferior: float, z_superior: float, discos: int
) -> None:
    """Calcula o volume pela fórmula, por Pappus e (nos segmentos) por discos."""
    # Os planos precisam cortar a esfera: as alturas ficam entre −raio e raio.
    if z_inferior < -raio or z_superior > raio:
        z_inferior, z_superior = max(z_inferior, -raio), min(z_superior, raio)
        print(f"(Alturas trazidas para dentro da esfera: de {z_inferior:g} a {z_superior:g}.)")
    if solido == "Segmento de uma base":
        z_superior = raio   # o plano de cima é tangente no polo norte
    if z_inferior >= z_superior:
        print("Escolha z_inferior menor que z_superior.")
        return

    # As fórmulas da seção 4.
    h = z_superior - z_inferior
    a = math.sqrt(raio**2 - z_inferior**2)       # raio da base de baixo
    b = math.sqrt(raio**2 - z_superior**2)       # raio da base de cima
    volume_segmento = math.pi * h / 6 * (3 * a**2 + 3 * b**2 + h**2)
    volume_setor = 2 / 3 * math.pi * raio**2 * h
    corda = math.sqrt((a - b)**2 + h**2)
    volume_anel = math.pi * corda**2 * h / 6
    if solido.startswith("Segmento"):
        volume_formula = volume_segmento
    elif solido == "Setor esférico":
        volume_formula = volume_setor
    else:
        volume_formula = volume_anel

    # Conferência por Pappus: área da região × caminho do centroide.
    regiao = REGIOES[solido](raio, z_inferior, z_superior, resolucao=200)
    area, centroide = u3d.medir_area_e_centroide_2d(regiao)
    volume_pappus = area * 2 * math.pi * centroide[0]

    fig = u3d.criar_figura(titulo=f"{solido} (giro de 300°)")
    fig = u3d.desenhar_solido_de_revolucao(
        fig, regiao, angulo=300, cor=COR_SOLIDO, opacidade=0.3
    )

    if solido.startswith("Segmento"):
        # Conferência por discos: na altura z, o disco tem raio √(R² − z²).
        dz = h / discos
        soma_discos = 0
        alturas_dos_discos = []
        for i in range(discos):
            z_meio = z_inferior + (i + 0.5) * dz
            soma_discos = soma_discos + math.pi * (raio**2 - z_meio**2) * dz
            alturas_dos_discos.append(z_meio)
        pilha = [
            u3d.poligono_regular(40, raio=math.sqrt(raio**2 - z**2), centro=(0, 0, z - dz / 2))
            for z in alturas_dos_discos
        ]
        fig = u3d.desenhar_pilha(fig, pilha, espessura=dz, opacidade=0.6)
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("A região que gira e o seu centroide")
    grafico = desenhar_esfera_no_plano(grafico, raio, [z_inferior, z_superior])
    grafico = u3d.desenhar_poligono_2d(grafico, regiao, rotulo="região", cor=COR_PARTE)
    grafico = u3d.desenhar_curva(
        grafico, [0, centroide[0]], [centroide[1], centroide[1]],
        rotulo="distância do centroide ao eixo", cor=COR_EXTRA, tracejado=True,
    )
    grafico = u3d.marcar_ponto_no_grafico(
        grafico, centroide[0], centroide[1], rotulo="centroide", cor=COR_EXTRA
    )
    u3d.mostrar(grafico)

    print(f"h = {h:g}   a = {a:.4f}   b = {b:.4f}   corda = {corda:.4f}")
    print(f"Volume pela fórmula: {volume_formula:.4f}")
    print(f"Volume por Pappus:   {volume_pappus:.4f}   (área {area:.4f} × 2π × {centroide[0]:.4f})")
    if solido.startswith("Segmento"):
        print(f"Volume por {discos} discos: {soma_discos:.4f}")


interact(
    conferir_volume,
    solido=widgets.Dropdown(
        options=list(REGIOES), description="sólido",
        style={"description_width": "initial"},
    ),
    raio=widgets.FloatSlider(
        min=1, max=4, step=0.5, value=2, description="raio",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    z_inferior=widgets.FloatSlider(
        min=-4, max=4, step=0.25, value=0.5, description="z_inferior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    z_superior=widgets.FloatSlider(
        min=-4, max=4, step=0.25, value=1.5, description="z_superior",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    discos=widgets.IntSlider(
        min=5, max=60, step=5, value=20, description="discos",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='sólido', options=('Segmento de uma base', 'Segmento de duas bases'…

**Pare e pense**

1. Numa esfera de raio 5, calcule o volume do segmento de uma base de altura 2 e o volume do setor esférico correspondente (mesma calota).
2. Confira que o setor da pergunta 1 é a soma do segmento com um cone. Quais são o raio e a altura desse cone?

<details>
<summary>Ver respostas</summary>

1. Segmento: $\frac{\pi \cdot 4}{3}(15 - 2) = \frac{52\pi}{3} \approx 54{,}5$. Setor: $\frac{2}{3}\pi \cdot 25 \cdot 2 = \frac{100\pi}{3} \approx 104{,}7$.
2. O cone tem vértice no centro e base na base da calota: raio $a = 4$ e altura $3$. Volume $= \frac{1}{3}\pi \cdot 16 \cdot 3 = 16\pi = \frac{48\pi}{3}$. E $\frac{52\pi}{3} + \frac{48\pi}{3} = \frac{100\pi}{3}$. ✔
</details>

## Resumo

**Esfera de raio $R$; partes entre planos paralelos a distância $h$; raios das bases $a$ e $b$.**

**Superfícies**
- **Zona** (duas bases) e **calota** (uma base): $S = 2\pi R h$, só depende da altura.
- Calota: também $S = \pi(a^2 + h^2) = \pi\ell^2$ ($\ell$ = distância do polo à borda da base).
- **Fuso** (notebook 13): $\frac{\alpha}{360°} \cdot 4\pi R^2$.

**Sólidos**
- **Segmento de uma base:** $\dfrac{\pi h^2}{3}(3R - h) = \dfrac{\pi h}{6}(3a^2 + h^2)$.
- **Segmento de duas bases:** $\dfrac{\pi h}{6}(3a^2 + 3b^2 + h^2)$.
- **Setor esférico:** $\dfrac{2}{3}\pi R^2 h$.
- **Anel esférico:** $\dfrac{\pi \ell^2 h}{6}$ ($\ell$ = corda, $h$ = projeção da corda no eixo). Esfera furada com anel de altura $h$: $\dfrac{\pi h^3}{6}$, para qualquer esfera.
- **Cunha** (notebook 13): $\dfrac{\alpha}{360°} \cdot \dfrac{4}{3}\pi R^3$.

**Calota visível de uma altura $H$:** $h = \dfrac{RH}{R + H}$; fração visível $= \dfrac{h}{2R}$ (sempre menos da metade).

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` nos cálculos.

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Uma calota esférica tem uma única base.
# (b) A área de uma zona esférica depende apenas do raio da esfera e da altura da zona.
# (c) Numa mesma esfera, uma zona perto do polo e uma zona perto do equador, de mesma
#     altura, têm a mesma área.
# (d) O volume de um setor esférico é (2/3)πR²h.
# (e) A cunha esférica é limitada por dois planos paralelos.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": True, "c": True, "d": True, "e": False}
dicas_ex1 = {
    "a": "a calota é cortada por um único plano (seção 1 - Superfícies esféricas: zona e calota)",
    "b": "S = 2πRh não depende de onde a zona está (seção 2 - Área da zona e da calota)",
    "c": "a faixa perto do polo é mais estreita, mas mais inclinada (seção 2 - Área da zona e da calota)",
    "d": "volume do setor = (1/3) · 2πRh · R (seção 4 - Volumes dos sólidos esféricos)",
    "e": "a cunha é limitada por dois semiplanos que passam pelo eixo (seção 3 - Sólidos esféricos)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: numa esfera de raio 5, calcule:
# (a) a área de uma zona de altura 2;
# (b) a área de uma calota de altura 2;
# (c) o volume do segmento esférico de uma base de altura 2.
area_zona_ex2 = ...
area_calota_ex2 = ...
volume_segmento_ex2 = ...

print(area_zona_ex2, area_calota_ex2, volume_segmento_ex2)

In [ ]:
# Verificação
assert area_zona_ex2 is not ... and abs(area_zona_ex2 - 20 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (a), S = 2πRh (seção 2 - Área da zona e da calota)."
)
assert area_calota_ex2 is not ... and abs(area_calota_ex2 - 20 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (b), a calota também tem S = 2πRh "
    "(seção 2 - Área da zona e da calota)."
)
assert volume_segmento_ex2 is not ... and abs(volume_segmento_ex2 - 52 * math.pi / 3) <= 0.01, (
    "Tente de novo, revise: no item (c), volume do segmento de uma base = πh²(3R − h)/3 "
    "(seção 4 - Volumes dos sólidos esféricos)."
)
print("Certo!")

In [ ]:
# Exercício Básico 3: o setor esférico. Numa esfera de raio 5, calcule o volume do
# setor esférico cuja calota tem altura 2.
volume_setor_ex3 = ...

print(volume_setor_ex3)

In [ ]:
# Verificação
assert volume_setor_ex3 is not ... and abs(volume_setor_ex3 - 100 * math.pi / 3) <= 0.01, (
    "Tente de novo, revise: volume do setor = (2/3)πR²h "
    "(seção 4 - Volumes dos sólidos esféricos)."
)
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: o astronauta. A Estação Espacial Internacional está a
# cerca de 400 km de altitude, e o raio da Terra é R = 6371 km. Calcule:
# (a) a altura da calota visível, em km;
# (b) a porcentagem da superfície da Terra visível (um número entre 0 e 100).
altura_calota_ex4 = ...
porcentagem_ex4 = ...

print(altura_calota_ex4, porcentagem_ex4)

In [ ]:
# Verificação
assert altura_calota_ex4 is not ... and abs(altura_calota_ex4 - 6371 * 400 / 6771) <= 1, (
    "Tente de novo, revise: no item (a), h = RH/(R + H) "
    "(seção 2 - Área da zona e da calota)."
)
assert porcentagem_ex4 is not ... and abs(porcentagem_ex4 - 100 * 400 / (2 * 6771)) <= 0.01, (
    "Tente de novo, revise: no item (b), fração = h/(2R); multiplique por 100 "
    "(seção 2 - Área da zona e da calota)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: a fatia de laranja. Uma laranja esférica de raio 5 cm é
# cortada por dois planos paralelos: um passando pelo centro e outro a 3 cm do
# centro. Qual é o volume da fatia, em cm³?
volume_fatia_ex5 = ...

print(volume_fatia_ex5)

In [ ]:
# Verificação
assert volume_fatia_ex5 is not ... and abs(volume_fatia_ex5 - 66 * math.pi) <= 0.01, (
    "Tente de novo, revise: é um segmento de duas bases com a = 5, b = 4 e h = 3 "
    "(seção 4 - Volumes dos sólidos esféricos)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: a calota misteriosa. Uma calota esférica tem base de
# raio 4 e altura 2. Calcule:
# (a) o raio da esfera;
# (b) a área da calota.
raio_esfera_ex6 = ...
area_calota_ex6 = ...

print(raio_esfera_ex6, area_calota_ex6)

In [ ]:
# Verificação
assert raio_esfera_ex6 is not ... and abs(raio_esfera_ex6 - 5) <= 0.01, (
    "Tente de novo, revise: no item (a), use 2Rh = a² + h² "
    "(seção 2 - Área da zona e da calota)."
)
assert area_calota_ex6 is not ... and abs(area_calota_ex6 - 20 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (b), S = π(a² + h²) "
    "(seção 2 - Área da zona e da calota)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: o anel de guardanapo. Uma esfera é furada por um furo
# cilíndrico que passa pelo centro, e o anel que sobra tem 6 cm de altura (de z = −3
# a z = 3). Calcule o volume do anel por Pappus, usando u3d.regiao_meridiana_do_anel
# e u3d.medir_area_e_centroide_2d:
# (a) se a esfera tiver raio 5 cm;
# (b) se a esfera tiver raio 10 cm.
# (c) Os dois volumes são iguais? (True ou False)
regiao_raio_5 = ...   # complete aqui: a região que gera o anel na esfera de raio 5
regiao_raio_10 = ...  # complete aqui: a região que gera o anel na esfera de raio 10
...  # complete aqui: a área e o centroide de cada região

volume_raio_5_ex7 = ...
volume_raio_10_ex7 = ...
volumes_iguais_ex7 = ...

print(volume_raio_5_ex7, volume_raio_10_ex7, volumes_iguais_ex7)

In [ ]:
# Verificação
respostas_ex7 = [("a", volume_raio_5_ex7), ("b", volume_raio_10_ex7)]
for item, resposta in respostas_ex7:
    assert resposta is not ... and abs(resposta - 36 * math.pi) <= 0.05, (
        f"Tente de novo, revise: no item ({item}), a região que gera o anel vai de z = −3 "
        "a z = 3, e o volume por Pappus é área × 2π × (abscissa do centroide) "
        "(seção 4 - Volumes dos sólidos esféricos)."
    )
assert volumes_iguais_ex7 is True, (
    "Tente de novo, revise: no item (c), compare os dois volumes; a fórmula πh³/6 não "
    "depende do raio (seção 4 - Volumes dos sólidos esféricos)."
)
print("Certo!")

Depois de acertar, rode a célula abaixo para ver as duas esferas furadas lado a lado.

In [7]:
# As duas esferas furadas, com anel de altura 6: raio 5 (à esquerda) e raio 10.
fig = u3d.criar_figura(titulo="Duas esferas furadas, com anéis de mesma altura")
for raio_da_esfera, deslocamento in [(5, -9), (10, 9)]:
    regiao = u3d.regiao_meridiana_do_anel(raio_da_esfera, -3, 3)
    # Cada anel é desenhado numa figura própria, em volta do eixo, e depois empurrado
    # para o lado (somando o deslocamento a todas as coordenadas x) antes de entrar
    # na figura final: assim os dois ficam lado a lado, na mesma escala.
    anel = u3d.desenhar_solido_de_revolucao(
        u3d.criar_figura(), regiao, angulo=300, cor=COR_SOLIDO, opacidade=0.6
    )
    for traco in anel.data:
        traco.x = np.asarray(traco.x, dtype=float) + deslocamento
        fig.add_trace(traco)
u3d.mostrar(fig)
print("A esfera maior tem um furo mais largo; as duas diferenças se compensam exatamente.")

A esfera maior tem um furo mais largo; as duas diferenças se compensam exatamente.


In [ ]:
# Exercício Desafio 8: os satélites. Use R = 6371 km.
# (a) Que porcentagem da superfície da Terra um satélite geoestacionário, a 35 786 km
#     de altitude, consegue ver?
# (b) Para qual valor a porcentagem visível se aproxima quando a altitude cresce sem
#     parar? Teste altitudes enormes no laço abaixo antes de responder.
R = 6371
porcentagem_geo_ex8 = ...

for H in [1_000, 100_000, 10_000_000, 1_000_000_000]:
    ...  # complete aqui: calcule e imprima a porcentagem visível da altitude H

limite_ex8 = ...

print(porcentagem_geo_ex8, limite_ex8)

In [ ]:
# Verificação
assert porcentagem_geo_ex8 is not ... and abs(porcentagem_geo_ex8 - 100 * 35786 / (2 * 42157)) <= 0.1, (
    "Tente de novo, revise: no item (a), fração = H/(2(R + H)); multiplique por 100 "
    "(seção 2 - Área da zona e da calota)."
)
assert limite_ex8 is not ... and abs(limite_ex8 - 50) <= 0.01, (
    "Tente de novo, revise: no item (b), para H muito grande, R + H ≈ H, e a resposta é "
    "uma porcentagem (seção 2 - Área da zona e da calota)."
)
print("Certo!")

## Conexões

**Isso depende de:**
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) (Princípio de Cavalieri)
- [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (anticlépsidra, área por "pirâmides", fuso e cunha)
- [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (tronco de cone)
- [`0617a_superficies_e_solidos_de_revolucao.ipynb`](0617a_superficies_e_solidos_de_revolucao.ipynb) (Pappus-Guldin e discos)
- [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (relações métricas no triângulo retângulo)

**Isso será usado em:**
- 📌 Geometria Analítica (equação da esfera e de planos que a cortam)
- 📌 Limite, Integral e Derivada (áreas e volumes de revolução)

---

## Fim do módulo 06

Você terminou a **Geometria Espacial**! Começou com pontos, retas e planos, passou por paralelismo, perpendicularidade, diedros e triedros, estudou os poliedros e os sólidos clássicos, e chegou às partes da esfera, usando ao longo do caminho Cavalieri, semelhança, Pappus e somas de fatias.

Para revisar os termos do módulo, use o [glossário de Geometria Espacial](glossario.md).